# 40 综合评价扩展方法

> **模块**：综合评价 ｜ **数据**：`../data/textbook/14.D 宏观投资效益.txt`、`../data/textbook/14.D 医院工作质量统计指标.txt` ｜ **依赖**：numpy、scipy、scikit-learn、pandas

## 1. 问题背景

除 AHP 与 TOPSIS 外，DEA 用投入产出比评价决策单元效率，主成分法用降维合成综合指标，秩和比按秩次排序分档，三者适用于不同的数据条件。

## 2. 学习目标

- 理解 DEA 的效率值与投入产出导向
- 会用主成分构造综合评价得分
- 掌握秩和比的编秩与分档方法

## 3. 环境与数据

- 依赖：numpy、scipy、scikit-learn、pandas
- 数据：`../data/textbook/14.D 宏观投资效益.txt`、`../data/textbook/14.D 医院工作质量统计指标.txt`

## 4. 方法与实现

### 4.1 数据包络分析

**从最有利于决策单元的角度进行评价，从而避免了确定各指标在优先意义下的权重.**

- 决策单元(Decision Making Units, DMU): 每个DMU有m种投入和s种产出.
- 向量$X_j,\ Y_j$分别表示决策单元$j$的输入和输出向量; $v,\ u$分别表示输入,输出权值向量.
- 评价单元$j$的效率评价指数为: $h_j=(u^TY_j)/(v^TX_j),\ j=1,2,\cdots,n$

In [1]:
# 例14.4问题

import numpy as np
import scipy.optimize

school_name = np.array(['A', 'B', 'C', 'D', 'E', 'F'])
X = np.array([[89.39, 64.3],
              [86.25, 99],
              [108.13, 99.6],
              [106.38, 96],
              [62.40, 96.2],
              [47.19, 79.9]])

Y = np.array([[25.2, 223],
              [28.2, 287],
              [29.4, 317],
              [26.4, 291],
              [27.2, 295],
              [25.2, 222]])

# 转化成线性规划问题, 用'01 线性规划.ipynb'的方法解
DEA = []
# 产出和投入拼起来
A_ub = np.concatenate((Y, X), axis = 1)
b_ub = np.zeros(A_ub.shape[0])
for index in range(X.shape[0]):
    # scipy的这个函数是求最小值的, 所以要取负号
    c = -np.concatenate((Y[index], np.zeros(2)))
    A_eq = [np.concatenate((np.zeros(2), X[index]))]
    b_eq = [-1]
    bounds = [(0, None), (0, None), (None, 0), (None, 0)]

    res = scipy.optimize.linprog(c=c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    DEA.append(np.round(-res.fun, 5))

DEA = np.array(DEA)
DEA_effective = school_name[np.where(DEA == 1)]
print('DEA有效的学校为: ', DEA_effective)

DEA有效的学校为:  ['A' 'E' 'F']


### 4.2 主成分分析法

In [2]:
# 例14.6
# 结果和书上有些不吻合.

import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA

data = np.loadtxt('../data/textbook/14.D 宏观投资效益.txt')
data = np.divide(np.subtract(data, data.mean(axis=0)), data.std(axis=0, ddof=1))
# print(data)
# 也可以直接用sklearn的预处理函数, 但这里ddof=1
# data = preprocessing.scale(data)

'''
# 直接使用numpy处理的
# 计算关联系数矩阵
# np.cov中的bias=False表示除以N-1而不是N
R = np.cov(data.T, bias=False)
# print(R)
eigenvalues, eigenvectors = np.linalg.eig(R)
order = eigenvalues.argsort()[::-1]
eigenvalues_order = eigenvalues[order]
eigenvectors = eigenvectors[order]
print(eigenvalues)
print(eigenvectors)
# 信息贡献率
b = np.array([eigenvalues[i]/eigenvalues.sum() for i in range(eigenvalues.shape[0])])
print(b)
# 累积贡献率
alpha = np.array([b[:i+1].sum() for i in range(b.shape[0])])
print(alpha)
'''

# PCA训练器
pca = PCA(n_components=3, copy=True, whiten=False)
# 训练并降维
newX = pca.fit_transform(data)
print(newX)
# 各个特征的方差百分比
print(pca.explained_variance_ratio_)
scores = np.matmul(newX, pca.explained_variance_ratio_)
print(np.sort(scores)[::-1])
print(np.matmul(np.eye(3), pca.explained_variance_ratio_))

[[ 0.69884609 -1.63512183 -0.041472  ]
 [-1.05574935  0.38201416  0.61956771]
 [-0.92259585 -1.18347819  1.08543643]
 [ 1.5372859  -0.64516989  0.46616436]
 [-0.2090462  -0.43873761 -0.46598478]
 [-1.31597026 -0.60192008 -0.08432748]
 [ 0.60305148 -0.67053247  0.06342084]
 [ 1.72056024  0.05159737  0.30965331]
 [ 3.04304363  0.50439662 -0.69392477]
 [ 2.92158292  2.55278482  0.26370719]
 [ 1.13409815  0.57219128  0.22483092]
 [-0.27572554 -0.24052626 -0.34786366]
 [-1.29962169  0.50357039 -0.62069783]
 [-1.70175932  0.09027513 -1.45544512]
 [-3.26297321  1.41283957  0.43559689]
 [-2.31387308  0.98093884  0.28281   ]
 [ 0.69884609 -1.63512183 -0.041472  ]]
[0.62686568 0.23366991 0.07003612]
[ 2.44641809  1.97684214  1.11230384  0.86037738  0.84556334  0.22579077
  0.05309933  0.05309933 -0.25340965 -0.26619943 -0.52915571 -0.74049026
 -0.77886716 -0.97149317 -1.14761366 -1.20146483 -1.68480033]
[0.62686568 0.23366991 0.07003612]


### 4.3 秩和比综合评价法

In [3]:
# 编秩函数
def get_rank(data, columns, ascending=True, R=np.zeros(1)):
    if data.ndim == 1:
        tempdata = np.array(data)[:,None]
    if not R.any():
        R = np.zeros(tempdata.shape)
    for i in columns:
        arg = tempdata[:,i].argsort(axis=0)
        if not ascending:
            arg = arg[::-1]
        begin, end = 0, 0
        # 找从begin开始相同项, 用end标记最后一个相同项的下一个
        while begin < len(arg):
            while end < len(arg) and tempdata[arg[end]][i] == tempdata[arg[begin]][i]:
                end += 1
            for j in range(begin, end):
                R[arg[j]][i] = (begin + end + 1) / 2
            begin = end
    return R.reshape(data.shape)

In [5]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn import linear_model


# ----------------------------------------------------------------------
# 编秩函数：对指定列做「平均秩」编秩，结果可累加到外部传入的 R 中
#   ascending=True  → 值越小秩越大（低优 / 成本型指标）
#   ascending=False → 值越大秩越大（高优 / 效益型指标）
# ----------------------------------------------------------------------
def get_rank(data, columns, ascending=True, R=None):
    data = np.asarray(data, dtype=float)

    # 只有外部没传 R 时才新建，避免 UnboundLocalError
    if R is None:
        R = np.zeros(data.shape, dtype=float)

    for i in columns:
        col = data[:, i]
        # 统一转成「值越大越优」的方向来算秩
        order_val = col if ascending else -col

        sorted_vals = np.sort(order_val)
        ranks = np.empty(len(col), dtype=float)
        for j, v in enumerate(order_val):
            below = np.sum(sorted_vals < v)      # 严格排在前面的个数
            equal = np.sum(sorted_vals == v)     # 与自己并列的个数（含自己）
            ranks[j] = below + (equal + 1) / 2   # 平均秩
        R[:, i] = ranks

    return R


# ----------------------------------------------------------------------
# 1. 读取数据
# ----------------------------------------------------------------------
data = np.loadtxt('../data/textbook/14.D 医院工作质量统计指标.txt')
w = data[-1]          # 最后一行是各指标权重
data = data[:-1]      # 前面是 1983–1992 年的指标数据

n, m = data.shape     # n=10 个评价对象，m=6 个指标


# ----------------------------------------------------------------------
# 2. 编秩并求加权秩和比 WRSR
# ----------------------------------------------------------------------
R = np.zeros(data.shape)

# 效益型指标（值越大越好）：x2, x6
R = get_rank(data, columns=[1, 5], ascending=False)
# 成本型/低优指标（值越小越好）：x1, x3, x4, x5
R = get_rank(data, columns=[0, 2, 3, 4], ascending=True, R=R)

# 加权秩和比：R 的每行与权重向量 w 做内积，再除以 n
WRSR = np.matmul(R, w) / R.shape[0]


# ----------------------------------------------------------------------
# 3. 构建 R 与 WRSR 的数据框（对应课本表 14.19）
# ----------------------------------------------------------------------
R_WRSR_columns = ['x' + str(i + 1) for i in range(R.shape[1])] + ['WRSR']
R_WRSR_index = [str(i) for i in range(1983, 1993)]

R_WRSR = pd.DataFrame(
    np.c_[R, WRSR],
    columns=R_WRSR_columns,
    index=R_WRSR_index
)

# 按 WRSR 升序排列，供后续计算累积频率使用
R_WRSR_sorted = R_WRSR.sort_values(by='WRSR', ascending=True)
print(R_WRSR_sorted, '\n')


# ----------------------------------------------------------------------
# 4. 计算累积频率、概率单位 Probit、回归拟合值
# ----------------------------------------------------------------------
f_columns = ['f', 'cf', 'p', 'Probit', 'WRSRfit', '排序']
f = pd.DataFrame(
    np.c_[np.ones((R_WRSR_sorted.shape[0], 1)),               # f 列（固定为 1）
          np.arange(1, n + 1),                                # cf 列：累积频数 1..n
          np.zeros((R_WRSR_sorted.shape[0], len(f_columns) - 2))],
    index=R_WRSR_sorted.index,
    columns=f_columns
)

# 累积频率 p = cf / n
f['p'] = f['cf'] / n
# 最后一个累积频率做修正，避免 Probit 趋于无穷
f.loc[f.index[-1], 'p'] = 1 - 1 / (4 * n)

# 概率单位：Probit = Φ^{-1}(p) + 5
f['Probit'] = norm.isf(1 - f['p'], 0, 1) + 5

# 以 Probit 为自变量、WRSR 为因变量做线性回归
reg = linear_model.LinearRegression()
reg.fit(np.array(f['Probit']).reshape((n, 1)), R_WRSR_sorted['WRSR'])

coef = reg.coef_[0]
intercept = reg.intercept_
print(f'WRSR = {coef:.6f} * Probit + {intercept:.6f}\n')

# 回归拟合值
f['WRSRfit'] = f['Probit'] * coef + intercept

# 按 WRSRfit 从大到小排序（1 为最优）
f['排序'] = np.array(f['WRSRfit'].argsort())[::-1] + 1

print(f)

        x1    x2    x3    x4    x5    x6     WRSR
1984   8.0   2.0   4.5   6.0   5.0   2.0  0.35820
1985  10.0   3.5   1.0   1.0   1.0   5.5  0.35975
1983   6.0   1.0   8.0   7.5   9.0   5.5  0.45385
1986   9.0   3.5   3.0   7.5   8.0   3.0  0.47070
1988   4.0   5.0   4.5   2.0   2.0  10.0  0.50420
1992   1.5  10.0   2.0   4.0   4.0   1.0  0.55345
1989   5.0   7.0   7.0   3.0   3.0   9.0  0.63400
1987   7.0   6.0   9.0   5.0   7.0   8.0  0.68050
1991   3.0   8.0   6.0   9.0  10.0   5.5  0.71695
1990   1.5   9.0  10.0  10.0   6.0   5.5  0.76840 

WRSR = 0.144075 * Probit + -0.198611

        f    cf      p    Probit   WRSRfit  排序
1984  1.0   1.0  0.100  3.718448  0.337123  10
1985  1.0   2.0  0.200  4.158379  0.400506   9
1983  1.0   3.0  0.300  4.475599  0.446209   8
1986  1.0   4.0  0.400  4.746653  0.485261   7
1988  1.0   5.0  0.500  5.000000  0.521762   6
1992  1.0   6.0  0.600  5.253347  0.558263   5
1989  1.0   7.0  0.700  5.524401  0.597315   4
1987  1.0   8.0  0.800  5.841621  

In [6]:
def get_rank(data, columns, ascending=True, R=None):
    data = np.asarray(data, dtype=float)
    if R is None:
        R = np.zeros(data.shape, dtype=float)     # ← 只在 R 未传时创建

    for i in columns:
        col = data[:, i]
        # 用「平均秩」处理并列：值相同则秩取平均
        if ascending:                              # 值越小秩越大（成本型/低优）
            order_val = col
        else:                                      # 值越大秩越大（效益型/高优）
            order_val = -col

        sorted_vals = np.sort(order_val)
        ranks = np.empty(len(col), dtype=float)
        for j, v in enumerate(order_val):
            below = np.sum(sorted_vals < v)       # 排在前面的个数
            equal = np.sum(sorted_vals == v)      # 与自己并列的个数（含自己）
            ranks[j] = below + (equal + 1) / 2    # 平均秩
        R[:, i] = ranks

    return R

## 5. 小结

方法选择取决于指标性质与样本量；不同方法的排序结果应相互对照，而非只报单一结论。